# 📊 01. Exploration et Compréhension des Données
**Projet AutoML – Challenge Machine Learning (M1 IA, Le Mans Université)**

Ce notebook a un objectif unique : **comprendre en détail les 6 jeux de données** du serveur Skinner avant de coder l'AutoML.

### Questions auxquelles ce notebook répond :
1. Où sont les données et comment sont-elles organisées ?
2. Quelles sont les dimensions ($N$ exemples, $D$ variables) ?
3. Quels sont les formats des données (dense classique vs matrices creuses) ?
4. Quelles sont les tâches à résoudre (Régression, Classification binaire, Multiclasse, Multi-label) ?
5. À quoi ressemblent visuellement les cibles ?

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import scipy.sparse as sp
import matplotlib.pyplot as plt

# Répertoire officiel des données sur le cluster Skinner
CORPUS_DIR = "/info/corpus/ChallengeMachineLearning"
DATASETS = ["data_A", "data_B", "data_C", "data_D", "data_E", "data_F"]

print(f"✓ Répertoire des données : {CORPUS_DIR}")
print(f"✓ Jeux de données disponibles : {DATASETS}")

---
## 1. Fonction de chargement universelle

Chaque jeu de données est composé de 3 fichiers :
- `<nom>.data` : Les variables d'entrée ($X$). Peut être sous format classique (dense) ou creux sans étiquette (`indice:valeur`).
- `<nom>.type` : Le type des colonnes (`Numerical`, `Categorical`, `Binary`).
- `<nom>.solution` : Les cibles à prédire ($y$).

In [ ]:
def load_features(filepath, n_features=None):
    """
    Charge le fichier .data.
    Gère automatiquement les matrices denses (np.loadtxt)
    et les matrices creuses LibSVM (scipy.sparse.csr_matrix).
    """
    with open(filepath, 'r') as f:
        first_line = f.readline().strip()
        
    if ':' in first_line:
        # Format creux (ex: data_F)
        rows, cols, data = [], [], []
        with open(filepath, 'r') as f:
            for r, line in enumerate(f):
                for item in line.strip().split():
                    c, v = item.split(':')
                    rows.append(r)
                    cols.append(int(c))
                    data.append(float(v))
        n_samples = r + 1
        min_c, max_c = min(cols), max(cols)
        # Si les indices commencent à 1
        if min_c == 1:
            cols = [c - 1 for c in cols]
            n_cols = n_features if n_features else max_c
        else:
            n_cols = n_features if n_features else max_c + 1
        return sp.csr_matrix((data, (rows, cols)), shape=(n_samples, n_cols))
    else:
        # Format dense standard
        return np.loadtxt(filepath)

---
## 2. Tableau récapitulatif des 6 jeux de données

Scannons l'ensemble des jeux pour afficher leurs caractéristiques principales dans un tableau clair :

In [ ]:
def analyser_dataset(name):
    folder = os.path.join(CORPUS_DIR, name)
    data_path = os.path.join(folder, f"{name}.data")
    type_path = os.path.join(folder, f"{name}.type")
    sol_path = os.path.join(folder, f"{name}.solution")
    
    # Types des colonnes
    col_types = []
    if os.path.exists(type_path):
        with open(type_path, 'r') as f:
            col_types = [l.strip() for l in f if l.strip()]
            
    # Solution
    sol = np.loadtxt(sol_path)
    
    # Données X
    X = load_features(data_path, n_features=len(col_types) if col_types else None)
    is_sparse = sp.issparse(X)
    
    # Déduction du problème
    if sol.ndim == 2 and sol.shape[1] > 1:
        if np.all(np.sum(sol, axis=1) == 1) and np.array_equal(sol, sol.astype(bool)):
            task = f"Multiclasse One-Hot ({sol.shape[1]} classes)"
        else:
            task = f"Multi-Label ({sol.shape[1]} labels binaires)"
    else:
        u = len(np.unique(sol))
        if u == 2:
            task = "Classification Binaire"
        elif u < 20 and np.all(np.equal(np.mod(sol, 1), 0)):
            task = f"Classification Multiclasse ({u} classes)"
        else:
            task = "Régression continue"
            
    return {
        "Dataset": name,
        "Lignes (N)": X.shape[0],
        "Variables (D)": X.shape[1],
        "Format X": "Creux (Sparse CSR)" if is_sparse else "Dense",
        "Forme Solution": str(sol.shape),
        "Type de Problème": task,
        "Types Colonnes": str(dict(pd.Series(col_types).value_counts())) if col_types else "Non spécifié"
    }

synthese = pd.DataFrame([analyser_dataset(d) for d in DATASETS])
synthese

---
## 3. Visualisations graphiques des cibles

Regardons concrètement la forme des cibles ($y$) pour 4 problèmes représentatifs :

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 8))

# 1. data_B : Régression continue
sol_B = np.loadtxt(os.path.join(CORPUS_DIR, 'data_B', 'data_B.solution'))
axes[0, 0].hist(sol_B, bins=25, color='skyblue', edgecolor='black')
axes[0, 0].set_title('data_B : Régression (Distribution de prix réels)')
axes[0, 0].set_xlabel('Valeur cible')
axes[0, 0].set_ylabel('Nombre exemples')

# 2. data_D : Classification binaire
sol_D = np.loadtxt(os.path.join(CORPUS_DIR, 'data_D', 'data_D.solution'))
classes_D, counts_D = np.unique(sol_D, return_counts=True)
axes[0, 1].bar([f'Classe {int(c)}' for c in classes_D], counts_D, color=['salmon', 'lightgreen'], edgecolor='black')
axes[0, 1].set_title('data_D : Classification Binaire (0 vs 1)')
axes[0, 1].set_ylabel('Nombre exemples')

# 3. data_C : Multiclasse One-Hot (10 classes)
sol_C = np.loadtxt(os.path.join(CORPUS_DIR, 'data_C', 'data_C.solution'))
labels_C = np.argmax(sol_C, axis=1)
classes_C, counts_C = np.unique(labels_C, return_counts=True)
axes[1, 0].bar(classes_C, counts_C, color='orange', edgecolor='black')
axes[1, 0].set_title('data_C : Multiclasse (Répartition des 10 classes)')
axes[1, 0].set_xlabel('Classe (0 à 9)')
axes[1, 0].set_ylabel('Nombre exemples')

# 4. data_A : Multi-label (3 cibles binaires)
sol_A = np.loadtxt(os.path.join(CORPUS_DIR, 'data_A', 'data_A.solution'))
axes[1, 1].bar(['Label 0', 'Label 1', 'Label 2'], np.sum(sol_A, axis=0), color='purple', alpha=0.7, edgecolor='black')
axes[1, 1].set_title('data_A : Multi-label (Fréquence par label)')
axes[1, 1].set_ylabel('Nombre de 1')

plt.tight_layout()
plt.show()

---
## 4. Ce qu'il faut retenir pour notre code AutoML

1. **Deux formats d'entrée :** On a du dense standard, mais aussi du creux (`data_F` avec 61 188 variables). Notre code doit manipuler `scipy.sparse` sans le convertir en tableau plein en RAM.
2. **Deux grandes familles :** 
   - La **Régression** (ex: `data_B`) ➡️ Modèles de régression (`RandomForestRegressor`, `Ridge`).
   - La **Classification** (ex: `data_D`, `data_C`, `data_F`) ➡️ Modèles de classification (`RandomForestClassifier`, `LogisticRegression`).
3. **Format des prédictions (`predict`) :** Si le jeu est encodé en One-Hot (ex: `data_C`), notre fonction `predict` doit obligatoirement renvoyer du One-Hot !